<a href="https://colab.research.google.com/github/francescoladu/RetrievalSphere/blob/main/RetrievalSphere.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Let's see how this thing works

In [ ]:
%pip install -q \
    ir-datasets==0.6.3 \
    ir-measures==0.4.3 \
    PyStemmer==3.1.0 \
    numpy==2.2.6 \
    pandas==2.2.3 \
    scipy==1.17.1 \
    tqdm==4.70.1 \
    psutil==7.2.2 \
    matplotlib==3.11.2

In [ ]:
import re
import unicodedata
from typing import Callable
from functools import lru_cache
from nltk.stem.snowball import SnowballStemmer
import subprocess
from pathlib import Path

ACTIVE_NORMALIZATION_LANGUAGES = ("en", "it")

STEM_CACHE_SIZE = 250_000

In [ ]:
# Local Colab storage
DATA_DIR = Path("/content/retrievalsphere/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

COLLECTION_URL = (
    "https://msmarco.z22.web.core.windows.net/"
    "msmarcoranking/collection.tar.gz"
)

ARCHIVE_PATH = DATA_DIR / "collection.tar.gz"
COLLECTION_PATH = DATA_DIR / "collection.tsv"


# Download only if necessary
if not ARCHIVE_PATH.exists():
    print("Downloading MS MARCO passage collection...")

    subprocess.run(
        [
            "wget",
            "-c",
            COLLECTION_URL,
            "-O",
            str(ARCHIVE_PATH),
        ],
        check=True,
    )
else:
    print("Archive already downloaded.")


# Extract only if necessary
if not COLLECTION_PATH.exists():
    print("Extracting collection...")

    subprocess.run(
        [
            "tar",
            "-xzf",
            str(ARCHIVE_PATH),
            "-C",
            str(DATA_DIR),
        ],
        check=True,
    )
else:
    print("Collection already extracted.")


print(f"Collection path: {COLLECTION_PATH}")

Extracting collection...
Collection path: /content/retrievalsphere/data/collection.tsv


In [ ]:

EXACT_EXPANSIONS_BY_LANG = {
    "en": {
        "can't": "cannot",
        "won't": "will not",
        "shan't": "shall not",
        "don't": "do not",
        "doesn't": "does not",
        "didn't": "did not",
        "isn't": "is not",
        "aren't": "are not",
        "wasn't": "was not",
        "weren't": "were not",
        "haven't": "have not",
        "hasn't": "has not",
        "hadn't": "had not",
        "couldn't": "could not",
        "wouldn't": "would not",
        "shouldn't": "should not",
        "mustn't": "must not",

        "i'm": "i am",

        "you're": "you are",
        "we're": "we are",
        "they're": "they are",

        "i've": "i have",
        "you've": "you have",
        "we've": "we have",
        "they've": "they have",

        "i'll": "i will",
        "you'll": "you will",
        "we'll": "we will",
        "they'll": "they will",

        "there's": "there is",
        "that's": "that is",
        "what's": "what is",
        "where's": "where is",
        "who's": "who is",

        "e.g.": "for example",
        "i.e.": "that is",
        "etc.": "etcetera",
    },

    "it": {
        "po'": "poco",
        "com'è": "come è",
        "cos'è": "cosa è",
        "ecc.": "eccetera",
        "dott.": "dottore",
    },

    "fr": {
        "c'est": "ce est",
        "n'est": "ne est",
        "etc.": "etcetera",
    },

    "es": {
        "p.ej.": "por ejemplo",
        "aprox.": "aproximadamente",
    },
}


PREFIX_EXPANSIONS_BY_LANG = {
    "it": (
        (r"(?<!\w)un'(?=\w)", "una "),
    ),

    "fr": (
        (r"(?<!\w)j'(?=\w)", "je "),
        (r"(?<!\w)qu'(?=\w)", "que "),
        (r"(?<!\w)n'(?=\w)", "ne "),
        (r"(?<!\w)c'(?=\w)", "ce "),
    ),
}

In [ ]:
def _build_exact_expansions(languages):
    expansions = {}

    for language in languages:
        expansions.update(
            EXACT_EXPANSIONS_BY_LANG.get(language, {})
        )

    return expansions


EXACT_EXPANSIONS = _build_exact_expansions(
    ACTIVE_NORMALIZATION_LANGUAGES
)


if EXACT_EXPANSIONS:
    _EXPANSION_RE = re.compile(
        r"(?<!\w)(?:"
        + "|".join(
            re.escape(key)
            for key in sorted(
                EXACT_EXPANSIONS,
                key=len,
                reverse=True
            )
        )
        + r")(?!\w)",
        flags=re.UNICODE,
    )
else:
    _EXPANSION_RE = None


_PREFIX_RULES = [
    (re.compile(pattern, flags=re.UNICODE), replacement)
    for language in ACTIVE_NORMALIZATION_LANGUAGES
    for pattern, replacement
    in PREFIX_EXPANSIONS_BY_LANG.get(language, ())
]

In [ ]:
def normalize_surface(text: str) -> str:
    """
    Basic surface normalization.

    Steps:
    1. Unicode NFKC normalization
    2. Case normalization
    """

    text = unicodedata.normalize("NFKC", text)

    text = text.casefold()

    return text


def expand_known_forms(text: str) -> str:
    """
    Expand known contractions, clitics and abbreviations.
    Unknown/ambiguous forms are left unchanged.
    """

    for pattern, replacement in _PREFIX_RULES:
        text = pattern.sub(replacement, text)

    if _EXPANSION_RE is not None:
        text = _EXPANSION_RE.sub(
            lambda match: EXACT_EXPANSIONS[match.group(0)],
            text
        )

    return text

In [ ]:
URL_PATTERN = (
    r"(?:https?://|www\.)"
    r"[^\s<>{}\[\]\"']+"
)

EMAIL_PATTERN = (
    r"[^\s@]+"
    r"@"
    r"[^\s@]+\.[^\s@]+"
)
DATE_PATTERN = (
    r"\b\d{1,4}[/-]\d{1,2}[/-]\d{1,4}\b"
)

PRICE_PATTERN = (
    r"[$€£¥]\s?\d+(?:[.,]\d+)*"
)

DECIMAL_PATTERN = (
    r"\b\d+(?:[.,]\d+)+\b"
)

ACRONYM_PATTERN = (
    r"\b(?:[^\W\d_]{1,4}\.){2,}"
    r"[^\W\d_]{0,4}\.?"
)

AMPERSAND_PATTERN = (
    r"\b[^\W_]+(?:&[^\W_]+)+\b"
)

HASHTAG_PATTERN = (
    r"#[^\W_]+(?:_[^\W_]+)*"
)

WORD_PATTERN = (
    r"[^\W_]+(?:'[^\W_]+)*"
)


_TOKEN_RE = re.compile(
    "|".join([
        rf"(?P<url>{URL_PATTERN})",
        rf"(?P<email>{EMAIL_PATTERN})",
        rf"(?P<date>{DATE_PATTERN})",
        rf"(?P<price>{PRICE_PATTERN})",
        rf"(?P<decimal>{DECIMAL_PATTERN})",
        rf"(?P<acronym>{ACRONYM_PATTERN})",
        rf"(?P<ampersand>{AMPERSAND_PATTERN})",
        rf"(?P<hashtag>{HASHTAG_PATTERN})",
        rf"(?P<word>{WORD_PATTERN})",
    ]),
    flags=re.UNICODE,
)

In [ ]:
_TRAILING_URL_PUNCTUATION = ".,;:!?)]}"


def _clean_url(token: str) -> str:
    token = token.rstrip(_TRAILING_URL_PUNCTUATION)
    token = re.sub(r"^https?://", "", token)
    token = re.sub(r"^www\.", "", token)
    return token.rstrip("/")


_CANONICALIZERS: dict[str, Callable[[str], str]] = {
    "url": _clean_url,
    "email": lambda t: t.rstrip(_TRAILING_URL_PUNCTUATION),
    "date": lambda t: re.sub(r"[/-]", "-", t),
    "price": lambda t: re.sub(r"\s+", "", t),
    "acronym": lambda t: t.replace(".", ""),
    "ampersand": lambda t: t.replace("&", ""),
    "hashtag": lambda t: t[1:],
}


def _canonicalize_token(token_type: str, token: str) -> str:
    transform = _CANONICALIZERS.get(token_type)
    return transform(token) if transform else token

In [ ]:
def tokenize(text: str) -> list[str]:
    """
    Tokenize a string using our custom rules.

    Pipeline:
        surface normalization
        -> contraction/abbreviation expansion
        -> regex tokenization
        -> special-token canonicalization
    """

    if not text:
        return []

    text = normalize_surface(text)
    text = expand_known_forms(text)

    tokens = []

    for match in _TOKEN_RE.finditer(text):
        token_type = match.lastgroup
        token = match.group(0)

        token = _canonicalize_token(
            token_type,
            token,
        )

        if token:
            tokens.append(token)

    return tokens

In [ ]:
_ENGLISH_STEMMER = SnowballStemmer("english")


@lru_cache(maxsize=STEM_CACHE_SIZE)
def stem_word(token: str) -> str:
    return _ENGLISH_STEMMER.stem(token)


def is_stemmable(token: str) -> bool:
    """
    Stem only alphabetic tokens.

    Therefore we do NOT stem:
        URLs
        emails
        prices
        dates
        decimal numbers

    Acronyms such as 'phd' are alphabetic but normally remain
    unchanged anyway.
    """
    return token.isalpha()

In [ ]:
def process_text(
    text: str
) -> list[str]:
    """
    Order:
        1. case / Unicode normalization
        2. contraction and abbreviation normalization
        3. punctuation-aware tokenization
        4. stop-word removal
        5. stemming
    """

    tokens = tokenize(text)

    tokens = [
        token
        for token in tokens
        if token not in STOPWORDS
    ]

    tokens = [
        stem_word(token)
        if is_stemmable(token)
        else token
        for token in tokens
    ]

    return tokens

In [ ]:
assert tokenize(
    "There's no way I can't do it."
) == [
    "there",
    "is",
    "no",
    "way",
    "i",
    "cannot",
    "do",
    "it",
]


assert tokenize(
    "I can’t do it."
) == [
    "i",
    "cannot",
    "do",
    "it",
]


assert tokenize(
    "Un'amica è arrivata."
) == [
    "una",
    "amica",
    "è",
    "arrivata",
]


assert tokenize(
    "Ph.D. and m.p.h."
) == [
    "phd",
    "and",
    "mph",
]


assert tokenize(
    "AT&T and R&D"
) == [
    "att",
    "and",
    "rd",
]

assert tokenize(
    "It costs $45.55."
) == [
    "it",
    "costs",
    "$45.55",
]


assert tokenize(
    "Dates: 01/02/06 and 01-02-06."
) == [
    "dates",
    "01-02-06",
    "and",
    "01-02-06",
]


assert tokenize(
    "Visit http://www.unipi.it."
) == [
    "visit",
    "unipi.it",
]


assert tokenize(
    "Visit https://www.unipi.it/"
) == [
    "visit",
    "unipi.it",
]


assert tokenize(
    "Follow #sigir."
) == [
    "follow",
    "sigir",
]

assert tokenize(
    "Contact name.surname@unipi.it."
) == [
    "contact",
    "name.surname@unipi.it",
]


assert tokenize(
    "Caffè, déjà-vu — naïve!"
) == [
    "caffè",
    "déjà",
    "vu",
    "naïve",
]


assert tokenize("") == []
assert process_text(
    "",
    remove_stopwords=False,
    apply_stemming=False,
) == []


print("All tokenizer regression tests passed.")